# Evaluating the transferred cell cycle score in gene-knockout cells

Explore and QC the scores produced by `19_run_ops0077_cc_phase_transfer_to_new_cells.ipynb` for the CCNB1, MCM3 and CENPC knockout cells, against the NTC reference.

**Stated expectation from the experiment design:** CCNB1 → G1, MCM3 → S, CENPC → G2/M. This is used for orientation only; nothing is tuned to it. (Note that loss of cyclin B1 is classically expected to arrest cells in G2/M, so a G2/M shift for CCNB1 would not by itself be a failure of the score.)

**What to look at**
1. Do the knockout cells lie on the NTC cycle at all? (profiles, distance to the curve)  → is the score meaningful for them?
2. Does the phase distribution shift relative to NTC, and in which sectors? (with bootstrap intervals)
3. Is the result consistent across guides, wells, and the two independent scores (stain vs phase contrast)?
4. Does it agree with the file's own gating (`ref_cycle_stage`)?
5. Is the DNA content on the same reference as the NTC? (G1 peak check)

Sectors (G1, S, G2/M, M-to-G1 exit) were fixed from the NTC curve before looking at any knockout cell.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, gaussian_kde

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
ROOT = Path("..").resolve() if (Path("..") / "reports").exists() else Path(".").resolve()
sys.path.insert(0, str(ROOT / "scripts/ops0077_cc_transfer"))
import cc_transfer_lib as L

REF_DIR = ROOT / "reports/ops0077_cc_transfer/reference"
SC_DIR = ROOT / "reports/ops0077_cc_transfer/ko3_scores"
EV_DIR = ROOT / "reports/ops0077_cc_transfer/ko3_evaluation"      # figures and tables from evaluate_cc_transfer.py
ref = L.load_reference(REF_DIR / "ntc_cc_reference.json")
ntc = pd.read_csv(REF_DIR / "ntc_reference_scores.csv", index_col=0)
prof = pd.read_csv(REF_DIR / "ntc_profiles_by_phase.csv")
ko = pd.read_csv(SC_DIR / "cc_phase_new_cells.csv", index_col=0)
SECT = L.SECTORS
EXPECTED = {"CCNB1": "G1", "MCM3": "S", "CENPC": "G2/M"}
genes = [g for g in EXPECTED if g in set(ko["gene"])]
COL = {"NTC": "0.35", "CCNB1": "#1f77b4", "MCM3": "#d62728", "CENPC": "#2ca02c"}
RNG = np.random.default_rng(0)
print(len(ko), "knockout cells |", len(ntc), "NTC cells,", int(ntc["cc_phase_stain"].notna().sum()), "scored")
pd.DataFrame(ref["sector_intervals"], columns=["sector", "from", "to"])

## 1. Coverage

In [ ]:
cov = ko.groupby("gene").agg(cells=("cc_phase", "size"), markers_present=("has_markers", "sum"), dna_reliable=("dna_reliable", "sum"),
                             stain_score=("cc_phase_stain", lambda s: int(s.notna().sum())), phase_score=("cc_phase_phase", lambda s: int(s.notna().sum())),
                             off_ntc_cycle=("out_of_ntc_support", "sum"))
cov["fraction_with_stain"] = (cov["stain_score"] / cov["cells"]).round(3)
cov["off_ntc_cycle_%"] = (100 * cov["off_ntc_cycle"] / cov["cells"]).round(1)
cov  # about 1% of NTC cells are beyond the threshold by construction

## 2. Phase distribution per gene against NTC
Choose which score to show: `cc_phase_stain` (markers + DNA), `cc_phase_phase` (phase contrast, all cells) or `cc_phase` (best available).

In [ ]:
SCORE = "cc_phase_stain"
fig = plt.figure(figsize=(15, 4.6))
ax = fig.add_subplot(1, 2, 1, projection="polar")
def polar(ax, ph, c, lab, lw=2):
    h, e = np.histogram(2 * np.pi * ph[~np.isnan(ph)], bins=36, range=(0, 2 * np.pi), density=True)
    ax.plot(np.r_[(e[:-1] + e[1:]) / 2, (e[0] + e[1]) / 2 + 2 * np.pi], np.r_[h, h[0]], color=c, lw=lw, label=lab)
polar(ax, ntc["cc_phase_stain"].values, COL["NTC"], "NTC", 3)
for g in genes: polar(ax, ko.loc[ko.gene == g, SCORE].values, COL[g], g)
ax.set_theta_zero_location("N"); ax.set_theta_direction(-1); ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=8); ax.set_title(SCORE)
ax = fig.add_subplot(1, 2, 2); xs = np.linspace(0, 1, 400)
for name, ph in [("NTC", ntc["cc_phase_stain"].values)] + [(g, ko.loc[ko.gene == g, SCORE].values) for g in genes]:
    ph = ph[~np.isnan(ph)]; ax.plot(xs, gaussian_kde(np.r_[ph - 1, ph, ph + 1], bw_method=.06)(xs) * 3, color=COL[name], lw=3 if name == "NTC" else 1.8, label=name)
for s, x0, x1 in ref["sector_intervals"]:
    ax.axvspan(x0, x1, alpha=.07, color={"G1": "C0", "S": "C1", "G2/M": "C3", "M-to-G1 exit": "C2"}[s])
ax.set_xlabel("phase (0 and 1 are the same point)"); ax.legend(fontsize=8)
plt.tight_layout()

## 3. Sector fractions with bootstrap intervals, and shift against NTC

In [ ]:
def valid(s):
    s = pd.Series(s, dtype=object); return s[s.notna() & (s != "")]
def fracs(s):
    s = valid(s); return s.value_counts(normalize=True).reindex(SECT, fill_value=0).values, len(s)
def boot(s, n=2000):
    s = valid(s).values.astype(object); idx = RNG.integers(0, len(s), (n, len(s)))
    return np.percentile(np.array([[np.mean(s[i] == k) for k in SECT] for i in idx]), [2.5, 97.5], axis=0)

SECTOR_COL = {"cc_phase_stain": "sector_stain", "cc_phase_phase": "sector_phase", "cc_phase": "sector_final"}[SCORE]
rows = []
f0, n0 = fracs(ntc["sector"])
for g in genes:
    s = ko.loc[ko.gene == g, SECTOR_COL]; f, n = fracs(s); ci = boot(s)
    for i, sec in enumerate(SECT):
        rows.append(dict(gene=g, sector=sec, n=n, ntc=f0[i], gene_fraction=f[i], delta=f[i] - f0[i], ci_low=ci[0][i] - f0[i], ci_high=ci[1][i] - f0[i],
                         expected=EXPECTED[g] == sec))
shift = pd.DataFrame(rows)
display(shift.pivot(index="gene", columns="sector", values="delta").reindex(columns=SECT).round(3))
shift.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for gi, g in enumerate(genes):
    d = shift[shift.gene == g].set_index("sector").reindex(SECT)
    ax.errorbar(np.arange(4) + gi * .22 - .22, d["delta"], [d["delta"] - d["ci_low"], d["ci_high"] - d["delta"]], fmt="o", color=COL[g], capsize=3, label=g)
ax.axhline(0, c="k", lw=.8); ax.set_xticks(range(4)); ax.set_xticklabels(SECT); ax.set_ylabel("change in fraction vs NTC (95% CI)"); ax.legend()
for g, s in EXPECTED.items():
    if g in genes: ax.annotate("expected " + g, (SECT.index(s) + (genes.index(g) - 1) * .22, ax.get_ylim()[0]), fontsize=7, color=COL[g], ha="center")
plt.tight_layout()

An interval that excludes 0 means the sector fraction differs from NTC. Compare the sign and size of the change with the stated expectation. Because the knockouts and NTC share the same wells and tiles, a shift is not a plate effect, but guide efficiency and the time of fixation still matter.

## 4. Sensitivity to the sector boundaries
Sectors were defined from the NTC DNA profile (G1 < 25% of the DNA range, G2/M > 75%). Do the conclusions depend on that choice?

In [ ]:
def sectors_with(lo, hi):
    f = np.array(ref["sector_grid"]["dna_fraction_along_curve"]); peak = int(np.argmax(f)); rising = np.arange(len(f)) <= peak
    lab = np.empty(len(f), object); lab[f < lo] = "G1"; lab[(f >= lo) & (f < hi) & rising] = "S"; lab[f >= hi] = "G2/M"; lab[(f >= lo) & (f < hi) & ~rising] = "M-to-G1 exit"
    return lab
rows = []
for lo, hi in [(0.15, 0.85), (0.25, 0.75), (0.35, 0.65)]:
    lab = sectors_with(lo, hi)
    cut = lambda ph: np.where(np.isnan(ph), None, lab[np.minimum((np.nan_to_num(ph) * len(lab)).astype(int), len(lab) - 1)])
    f0_, _ = fracs(pd.Series(cut(ntc["cc_phase_stain"].values)))
    for g in genes:
        f_, n_ = fracs(pd.Series(cut(ko.loc[ko.gene == g, "cc_phase_stain"].values)))
        rows.append(dict(thresholds=f"{lo}/{hi}", gene=g, **{f"d_{s}": round(f_[i] - f0_[i], 3) for i, s in enumerate(SECT)}))
pd.DataFrame(rows)

## 5. Are the cells without a stain score different?
Cells lacking markers or a reliable DNA measurement only get the phase-contrast score. If they differ from the scored cells, the stain-score result could be selection-biased.

In [ ]:
ko["has_stain"] = ko["cc_phase_stain"].notna()
d = ko.groupby(["gene", "has_stain"])["sector_phase"].apply(lambda s: pd.Series(fracs(s)[0], index=SECT)).unstack().round(3)
display(d)   # phase-contrast sectors of cells with vs without a stain score
fig, ax = plt.subplots(1, len(genes), figsize=(5 * len(genes), 3.6), sharey=True)
for a, g in zip(np.atleast_1d(ax), genes):
    for hs, c in [(True, "C0"), (False, "C1")]:
        ph = ko.loc[(ko.gene == g) & (ko.has_stain == hs), "cc_phase_phase"].dropna().values
        a.hist(ph, bins=25, alpha=.55, density=True, color=c, label=f"{'with' if hs else 'without'} stain score (n={len(ph)})")
    a.set_title(g); a.set_xlabel("phase-contrast phase"); a.legend(fontsize=7)
plt.tight_layout()

## 6. Replicates: per guide and per well

In [ ]:
# guide = the 20-nt protospacer (the barcode column is truncated to 9 or 10 nt, which would split one guide in two)
ko["guide"] = ko["gene"] + "|" + ko["sgRNA"].astype(str)
g_tab = ko.groupby(["gene", "guide"])[SECTOR_COL].apply(lambda s: pd.Series(fracs(s)[0], index=SECT)).unstack()
g_tab["n"] = ko.groupby(["gene", "guide"])[SECTOR_COL].apply(lambda s: len(valid(s)))
display(g_tab[g_tab.n >= 10].round(2))
# heterogeneity of the sector composition across guides within a gene (chi-square on counts)
for g in genes:
    sub = ko[(ko.gene == g) & ko[SECTOR_COL].notna() & (ko[SECTOR_COL] != "")]
    ct = pd.crosstab(sub["guide"], sub[SECTOR_COL]); ct = ct[ct.sum(1) >= 10]
    if len(ct) > 1: print(f"{g}: {len(ct)} guides, chi-square p across guides = {chi2_contingency(ct)[1]:.3g}")

In [ ]:
# per-guide forest plot of the fraction in the stated expected sector, against the NTC fraction
fig, ax = plt.subplots(figsize=(8, 5)); y = 0; ticks = []
for g in genes:
    sub = g_tab.loc[g]; sub = sub[sub.n >= 10]; sec = EXPECTED[g]
    for guide, r in sub.iterrows():
        p, n = r[sec], r["n"]; se = np.sqrt(p * (1 - p) / n)
        ax.errorbar(p, y, xerr=1.96 * se, fmt="o", color=COL[g]); ticks.append((y, f"{guide}  (n={int(n)})")); y += 1
    ax.axvline(f0[SECT.index(sec)], color=COL[g], ls=":", lw=1)
    y += 1
ax.set_yticks([t[0] for t in ticks]); ax.set_yticklabels([t[1] for t in ticks], fontsize=7); ax.invert_yaxis()
ax.set_xlabel("fraction of cells in the stated expected sector (dotted: NTC)"); plt.tight_layout()

In [ ]:
w = ko.groupby(["gene", "well"])[SECTOR_COL].apply(lambda s: pd.Series(fracs(s)[0], index=SECT)).unstack().round(2)
w["n"] = ko.groupby(["gene", "well"])[SECTOR_COL].apply(lambda s: len(valid(s)))
nw = ntc.groupby("well")["sector"].apply(lambda s: pd.Series(fracs(s)[0], index=SECT)).unstack().round(2)
display(pd.concat([nw.assign(gene="NTC", n=ntc.groupby("well")["sector"].apply(lambda s: len(valid(s)))).reset_index().set_index(["gene", "well"]), w]))

## 7. Do the knockout cells follow the NTC cycle? Profiles and distance to the NTC curve

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
for a, (f, t) in zip(ax, [("z_dna", "DNA (z)"), ("z_phos", "phosphorylated Rb (z)"), ("z_p21", "p21 (z)"), ("z_pRb", "pRb (z)")]):
    a.fill_between(prof["phase_mid"], prof[f + "_q25"], prof[f + "_q75"], color="0.75", alpha=.5); a.plot(prof["phase_mid"], prof[f + "_q50"], c="0.2", lw=3, label="NTC")
    for g in genes:
        d = ko[(ko.gene == g) & ko["cc_phase_stain"].notna()]; b = pd.cut(d["cc_phase_stain"], np.linspace(0, 1, 16), include_lowest=True)
        m, n = d.groupby(b, observed=True)[f].median(), d.groupby(b, observed=True).size(); mid = (np.linspace(0, 1, 16)[:-1] + 1 / 30)[:len(m)]
        a.plot(mid[n.values >= 5], m.values[n.values >= 5], "o-", ms=3, c=COL[g], label=g)
    a.set_title(t); a.set_xlabel("phase")
ax[0].legend(fontsize=8); plt.tight_layout()

In [ ]:
q99 = ref["ntc_curve_dist_q99"]; xs = np.linspace(0, 8, 200)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for name, d in [("NTC", ntc["cc_curve_dist"].dropna())] + [(g, ko.loc[ko.gene == g, "cc_curve_dist"].dropna()) for g in genes]:
    ax[0].plot(xs, gaussian_kde(d)(xs), color=COL[name], lw=3 if name == "NTC" else 1.8, label=name)
ax[0].axvline(q99, c="k", ls="--"); ax[0].set_xlabel("distance to the NTC cycle curve"); ax[0].legend()
ko.groupby("gene")["out_of_ntc_support"].mean().reindex(genes).mul(100).plot.bar(ax=ax[1], color=[COL[g] for g in genes]); ax[1].axhline(1, c="k", ls="--")
ax[1].set_ylabel("% cells beyond the NTC 99th percentile (dashed: 1%)"); plt.tight_layout()

In [ ]:
# who are the cells off the NTC cycle? (compare their axes with the on-cycle cells)
off = ko[ko["out_of_ntc_support"]]
display(pd.concat([off[["z_dna", "z_phos", "z_p21", "z_pRb", "nuc_area_px"]].median().rename("off cycle"),
                   ko[~ko["out_of_ntc_support"] & ko["cc_phase_stain"].notna()][["z_dna", "z_phos", "z_p21", "z_pRb", "nuc_area_px"]].median().rename("on cycle")], axis=1).round(2))

## 8. Two independent scores: stain vs phase contrast

In [ ]:
both = ko.dropna(subset=["cc_phase_stain", "cc_phase_phase"]).copy(); both["err"] = L.circ_diff(both["cc_phase_phase"], both["cc_phase_stain"])
display(both.groupby("gene")["err"].agg(n="size", mae=lambda e: e.abs().mean(), within_0p10=lambda e: (e.abs() <= .1).mean()).round(3))
fig, ax = plt.subplots(1, len(genes), figsize=(5 * len(genes), 4))
for a, g in zip(np.atleast_1d(ax), genes):
    d = both[both.gene == g]; a.hexbin(d["cc_phase_stain"], d["cc_phase_phase"], gridsize=25, cmap="viridis", mincnt=1); a.set_title(g)
    a.set_xlabel("stain score"); a.set_ylabel("phase-contrast score")
plt.tight_layout()

If the two scores agree about as well in the knockouts as in the NTC cells (mean error about 0.1 turn), the phase-contrast model is not being fooled by a knockout-specific change in cell shape, and its result for the cells without a stain score can be trusted similarly.

## 9. Agreement with the gating columns shipped in the input file (comparison only)

In [ ]:
for rc in ["ref_cycle_stage", "ref_phase_predicted_stage"]:
    if rc in ko:
        d = ko[ko.sector_stain.notna() & (ko.sector_stain != "")]
        print(rc, "(rows) vs our stain sector (columns), row-normalised"); display(pd.crosstab(d[rc].astype(str), d["sector_stain"], normalize="index").reindex(columns=SECT, fill_value=0).round(2))
# the file's own labels, by gene (these were NOT used by our score)
display(pd.crosstab(ko["gene"], ko["ref_cycle_stage"].astype(str), normalize="index").round(2))

## 10. DNA content reference check (QC only)
The lowest-DNA (G1, 2N) peak exists in every population, so it should be at the same position in knockout and NTC cells of the same well. A shift would mean the DNA content of the new file is on another reference.

In [ ]:
xs = np.linspace(-1, 2, 300); rows = []
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
for a, w in zip(ax, sorted(ko["well"].unique())):
    for name, dna in [("NTC", ntc.loc[ntc.well == w, "dna_rel"].dropna().values)] + [(g, ko.loc[(ko.gene == g) & (ko.well == w), "dna_content"].dropna().values) for g in genes]:
        if len(dna) < 20: continue
        kd = gaussian_kde(dna, bw_method=.15)(xs); low = (xs > -.6) & (xs < .35)
        rows.append(dict(well=w, group=name, n=len(dna), g1_peak=round(float(xs[low][np.argmax(kd[low])]), 3)))
        a.plot(xs, kd, color=COL[name], lw=3 if name == "NTC" else 1.5, label=name)
    a.set_title(w); a.set_xlabel("relative DNA content (log2)"); a.legend(fontsize=7)
plt.tight_layout(); pd.DataFrame(rows).pivot(index="well", columns="group", values="g1_peak")

## 11. Figures and tables written by the script
`evaluate_cc_transfer.py` saves the same analyses as figures and CSV files in `reports/ops0077_cc_transfer/ko3_evaluation/` (`fig1` to `fig9`, `sector_shift_vs_ntc.csv`, `fractions_per_guide.csv`, ...). Re-run it after rescoring to refresh them.